# Предобработка данных: HR-аналитика (кейс 4)
## Датасет

Используемые файлы:
- `Employee.csv` — основная таблица сотрудников (1470 строк × 23 столбца)
- `PerformanceRating.csv` — таблица оценок производительности и удовлетворённости (6709 строк × 11 столбцов)
- `EducationLevel.csv` — справочник уровней образования (5 × 2)
- `RatingLevel.csv` — справочник уровней рейтинга (5 × 2)
- `SatisfiedLevel.csv` — справочник уровней удовлетворённости (5 × 2)

## Цель этапа
Провести предобработку: привести названия колонок к стандарту, преобразовать даты, оставить последний **валидный** опрос на сотрудника (дата опроса не раньше даты приема и не позже даты приема плюс стаж), пометить аномалии без исключения строк, объединить таблицы и создать новые признаки для последующего анализа текучести.

## Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd

Настройка подключения через гугл диск.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))

['Проект команды 3', 'Личное']


In [ ]:
print(os.listdir('/content/drive/MyDrive/Проект команды 3'))


['DimDate.txt', 'EducationLevel.csv', 'Employee.csv', 'PerformanceRating.csv', 'RatingLevel.csv', 'SatisfiedLevel.csv', 'Employee_cleaned.csv', 'v2_Проект_команды_3_Предобработка_данных.ipynb', 'Проект команды 3. Предобработка данных.ipynb', 'Проект команды 3_Гипотеза 4.ipynb', 'V4_Проект_команды_3_Предобработка_данных.ipynb', 'V3_Employee_cleaned.csv', 'V3_Проект_команды_3_Предобработка_данных.ipynb']


## Загрузка исходных данных
Загружаем все CSV-файлы из папки проекта.

In [ ]:
path = '/content/drive/MyDrive/Проект команды 3/'
education = pd.read_csv(path + 'EducationLevel.csv')
employee = pd.read_csv(path + 'Employee.csv')
performance = pd.read_csv(path + 'PerformanceRating.csv')
rating = pd.read_csv(path + 'RatingLevel.csv')
satisfied = pd.read_csv(path + 'SatisfiedLevel.csv')

## Первичный осмотр таблиц
Смотрим структуру, первые строки и размеры каждой таблицы через `.info()`, `.head()`, `.shape`.

In [ ]:
education.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   EducationLevelID  5 non-null      int64 
 1   EducationLevel    5 non-null      object
dtypes: int64(1), object(1)
memory usage: 212.0+ bytes


In [ ]:
education.head()

,EducationLevelID,EducationLevel
0,1,No Formal Qualifications
1,2,High School
2,3,Bachelors
3,4,Masters
4,5,Doctorate


In [ ]:
education.shape

(5, 2)

**Наблюдение по таблице Education:**
1. Справочник содержит 5 уникальных уровней образования, от High School до Doctorate.
2. Данные чистые, полные, без дублей и пропусков.

In [ ]:
employee.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   EmployeeID               1470 non-null   object
 1   FirstName                1470 non-null   object
 2   LastName                 1470 non-null   object
 3   Gender                   1470 non-null   object
 4   Age                      1470 non-null   int64 
 5   BusinessTravel           1470 non-null   object
 6   Department               1470 non-null   object
 7   DistanceFromHome (KM)    1470 non-null   int64 
 8   State                    1470 non-null   object
 9   Ethnicity                1470 non-null   object
 10  Education                1470 non-null   int64 
 11  EducationField           1470 non-null   object
 12  JobRole                  1470 non-null   object
 13  MaritalStatus            1470 non-null   object
 14  Salary                   1470 non-null  

In [ ]:
employee.head()

,EmployeeID,FirstName,LastName,Gender,Age,BusinessTravel,Department,DistanceFromHome (KM),State,Ethnicity,...,MaritalStatus,Salary,StockOptionLevel,OverTime,HireDate,Attrition,YearsAtCompany,YearsInMostRecentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,Divorced,102059,1,No,2012-01-03,No,10,4,9,7
1,CBCB-9C9D,Leonerd,Aland,Male,38,Some Travel,Sales,23,CA,White,...,Single,157718,0,Yes,2012-01-04,No,10,6,10,0
2,95D7-1CE9,Ahmed,Sykes,Male,43,Some Travel,Human Resources,29,CA,Asian or Asian American,...,Married,309964,1,No,2012-01-04,No,10,6,10,8
3,47A0-559B,Ermentrude,Berrie,Non-Binary,39,Some Travel,Technology,12,IL,White,...,Married,293132,0,No,2012-01-05,No,10,10,10,0
4,42CC-040A,Stace,Savege,Female,29,Some Travel,Human Resources,29,CA,White,...,Single,49606,0,No,2012-01-05,Yes,6,1,1,6


In [ ]:
employee.shape

(1470, 23)

**Наблюдение по таблице Employee:**
1. Gender включает значение `Non-Binary` — данные учитывают больше двух категорий пола.
2. Salary сильно разбросан: от ~49 606 до ~309 964 в первых пяти строках.
3. Attrition уже встречается как `Yes` (строка 4) и `No` (остальные), преобразуем в флаг.
4. HireDate представлен строками (2012-01-03), а не датами — потребуется приведение к `datetime`.

In [ ]:
performance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6709 entries, 0 to 6708
Data columns (total 11 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   PerformanceID                    6709 non-null   object
 1   EmployeeID                       6709 non-null   object
 2   ReviewDate                       6709 non-null   object
 3   EnvironmentSatisfaction          6709 non-null   int64 
 4   JobSatisfaction                  6709 non-null   int64 
 5   RelationshipSatisfaction         6709 non-null   int64 
 6   TrainingOpportunitiesWithinYear  6709 non-null   int64 
 7   TrainingOpportunitiesTaken       6709 non-null   int64 
 8   WorkLifeBalance                  6709 non-null   int64 
 9   SelfRating                       6709 non-null   int64 
 10  ManagerRating                    6709 non-null   int64 
dtypes: int64(8), object(3)
memory usage: 576.7+ KB


In [ ]:
performance.head()

,PerformanceID,EmployeeID,ReviewDate,EnvironmentSatisfaction,JobSatisfaction,RelationshipSatisfaction,TrainingOpportunitiesWithinYear,TrainingOpportunitiesTaken,WorkLifeBalance,SelfRating,ManagerRating
0,PR01,79F7-78EC,1/2/2013,5,4,5,1,0,4,4,4
1,PR02,B61E-0F26,1/3/2013,5,4,4,1,3,4,4,3
2,PR03,F5E3-48BB,1/3/2013,3,4,5,3,2,3,5,4
3,PR04,0678-748A,1/4/2013,5,3,2,2,0,2,3,2
4,PR05,541F-3E19,1/4/2013,5,2,3,1,0,4,4,3


In [ ]:
performance.shape

(6709, 11)

**Наблюдение по таблице Performance:**
1. ReviewDate имеет тип `object`, нужно преобразовать в `datetime`.
2. SelfRating / ManagerRating — могут быть пропуски или некорректные значения.

In [ ]:
rating.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   RatingID     5 non-null      int64 
 1   RatingLevel  5 non-null      object
dtypes: int64(1), object(1)
memory usage: 212.0+ bytes


In [ ]:
rating.head()

,RatingID,RatingLevel
0,1,Unacceptable
1,2,Needs Improvement
2,3,Meets Expectation
3,4,Exceeds Expectation
4,5,Above and Beyond


In [ ]:
rating.shape

(5, 2)

**Наблюдение по таблице Rating:**
1. Справочник содержит 5 категорий оценки эффекивности.
2. Данные чистые, полные, без дублей и пропусков.

In [ ]:
satisfied.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   SatisfactionID     5 non-null      int64 
 1   SatisfactionLevel  5 non-null      object
dtypes: int64(1), object(1)
memory usage: 212.0+ bytes


In [ ]:
satisfied.head()

,SatisfactionID,SatisfactionLevel
0,1,Very Dissatisfied
1,2,Dissatisfied
2,3,Neutral
3,4,Satisfied
4,5,Very Satisfied


In [ ]:
satisfied.shape

(5, 2)

**Наблюдение по таблице Satisfied:**
1. Справочник содержит 5 категорий удовлетворенности.
2. Данные чистые, полные, без дублей и пропусков.

## Стандартизация названий колонок
Приводим имена столбцов к нижнему регистру, заменяем пробелы, слеши и дефисы на нижнее подчеркивание.

In [ ]:
def clean_columns(df):
    return df.rename(columns=lambda name: name.lower().replace(' ', '_').replace('/', '_').replace('-', '_'))

education_clean = clean_columns(education)
employee_clean = clean_columns(employee)
performance_clean = clean_columns(performance)
rating_clean = clean_columns(rating)
satisfied_clean = clean_columns(satisfied)

In [ ]:
print(education_clean.columns.tolist())

['educationlevelid', 'educationlevel']


In [ ]:
print(employee_clean.columns.tolist())

['employeeid', 'firstname', 'lastname', 'gender', 'age', 'businesstravel', 'department', 'distancefromhome_(km)', 'state', 'ethnicity', 'education', 'educationfield', 'jobrole', 'maritalstatus', 'salary', 'stockoptionlevel', 'overtime', 'hiredate', 'attrition', 'yearsatcompany', 'yearsinmostrecentrole', 'yearssincelastpromotion', 'yearswithcurrmanager']


In [ ]:
print(performance_clean.columns.tolist())

['performanceid', 'employeeid', 'reviewdate', 'environmentsatisfaction', 'jobsatisfaction', 'relationshipsatisfaction', 'trainingopportunitieswithinyear', 'trainingopportunitiestaken', 'worklifebalance', 'selfrating', 'managerrating']


In [ ]:
print(rating_clean.columns.tolist())

['ratingid', 'ratinglevel']


In [ ]:
print(satisfied_clean.columns.tolist())

['satisfactionid', 'satisfactionlevel']


## Преобразование дат
Переводим текстовые даты в тип `datetime64`.

In [ ]:
employee_clean['hiredate'] = pd.to_datetime(employee_clean['hiredate'], format='%Y-%m-%d')
performance_clean['reviewdate'] = pd.to_datetime(performance_clean['reviewdate'], format='%m/%d/%Y')

In [ ]:
print(employee_clean[['employeeid', 'hiredate']].head())

  employeeid   hiredate
0  3012-1A41 2012-01-03
1  CBCB-9C9D 2012-01-04
2  95D7-1CE9 2012-01-04
3  47A0-559B 2012-01-05
4  42CC-040A 2012-01-05


In [ ]:
print(performance_clean[['performanceid', 'reviewdate']].head())

  performanceid reviewdate
0          PR01 2013-01-02
1          PR02 2013-01-03
2          PR03 2013-01-03
3          PR04 2013-01-04
4          PR05 2013-01-04


In [ ]:
print(employee_clean.dtypes[['hiredate']])
print(performance_clean.dtypes[['reviewdate']])

hiredate    datetime64[ns]
dtype: object
reviewdate    datetime64[ns]
dtype: object


## Отбор последнего валидного опроса на сотрудника
Дата опроса (`reviewdate`) должна быть **не раньше** даты приема (`hiredate`) и **не позже** даты приема плюс стаж работы (`yearsatcompany` лет) — это приближенная дата увольнения или текущая дата. Сначала соединяем ревью с данными о сотруднике, фильтруем невалидные даты, а затем берём последний опрос на сотрудника.

In [ ]:
performance_with_emp = performance_clean.merge(
    employee_clean[['employeeid', 'hiredate', 'yearsatcompany']],
    on='employeeid',
    how='left'
)

performance_with_emp['hire_plus_tenure'] = performance_with_emp['hiredate'] + pd.to_timedelta(
    performance_with_emp['yearsatcompany'] * 365, unit='D'
)

valid_mask = (
    (performance_with_emp['reviewdate'] >= performance_with_emp['hiredate']) &
    (performance_with_emp['reviewdate'] <= performance_with_emp['hire_plus_tenure'])
)

performance_valid = performance_with_emp[valid_mask].copy()
print(f'Всего ревью: {len(performance_with_emp)}')
print(f'Валидных ревью (дата в допустимом диапазоне): {len(performance_valid)}')
print(f'Отброшено ревью с некорректной датой: {len(performance_with_emp) - len(performance_valid)}')

performance_sorted = performance_valid.sort_values(['employeeid', 'reviewdate'])
last_rating = performance_sorted.drop_duplicates('employeeid', keep='last')

last_rating = last_rating.drop(columns=['hiredate', 'yearsatcompany', 'hire_plus_tenure'], errors='ignore')

employees_without_rating = set(employee_clean['employeeid']) - set(last_rating['employeeid'])
print(f'Сотрудников без валидного опроса: {len(employees_without_rating)}')
performance_with_emp = performance_clean.merge(
    employee_clean[['employeeid', 'hiredate', 'yearsatcompany']],
    on='employeeid',
    how='left'
)

Всего ревью: 6709
Валидных ревью (дата в допустимом диапазоне): 4386
Отброшено ревью с некорректной датой: 2323
Сотрудников без валидного опроса: 381


In [ ]:
print(f'Всего сотрудников: {len(employee_clean)}')
print(f'Сотрудников с валидным опросом: {len(last_rating)}')
print(f'Сотрудников без опроса: {len(employees_without_rating)}')
print(f'Размер last_rating: {last_rating.shape}')

Всего сотрудников: 1470
Сотрудников с валидным опросом: 1089
Сотрудников без опроса: 381
Размер last_rating: (1089, 11)


## Объединение таблиц
Соединяем основную таблицу сотрудников с последними валидными оценками и справочниками. Связь 1:1 по `employeeid`.

In [ ]:
df = employee_clean.merge(
    last_rating,
    on='employeeid',
    how='left',
    validate='1:1'
)

df = df.merge(
    education_clean,
    left_on='education',
    right_on='educationlevelid',
    how='left'
)

df = df.merge(
    rating_clean,
    left_on='selfrating',
    right_on='ratingid',
    how='left'
)

df = df.merge(
    satisfied_clean,
    left_on='jobsatisfaction',
    right_on='satisfactionid',
    how='left'
)

print('Размер после объединения:', df.shape)

Размер после объединения: (1470, 39)


In [ ]:
print(df.head(10))

  employeeid    firstname     lastname      gender  age      businesstravel  \
0  3012-1A41     Leonelle        Simco      Female   30         Some Travel   
1  CBCB-9C9D      Leonerd        Aland        Male   38         Some Travel   
2  95D7-1CE9        Ahmed        Sykes        Male   43         Some Travel   
3  47A0-559B   Ermentrude       Berrie  Non-Binary   39         Some Travel   
4  42CC-040A        Stace       Savege      Female   29         Some Travel   
5  C219-6C2E  Clerkclaude      Hinkins        Male   34         Some Travel   
6  D906-B674          Uta       Melmar      Female   42          No Travel    
7  3C7D-86ED        Joyan       Brason      Female   40         Some Travel   
8  3D71-8DC2         Alix  Blazejewski        Male   38         Some Travel   
9  5476-CA0D       Kayley        Snoad      Female   31  Frequent Traveller   

        department  distancefromhome_(km) state  \
0            Sales                     27    IL   
1            Sales          

In [ ]:
print(df.dtypes)

employeeid                                 object
firstname                                  object
lastname                                   object
gender                                     object
age                                         int64
businesstravel                             object
department                                 object
distancefromhome_(km)                       int64
state                                      object
ethnicity                                  object
education                                   int64
educationfield                             object
jobrole                                    object
maritalstatus                              object
salary                                      int64
stockoptionlevel                            int64
overtime                                   object
hiredate                           datetime64[ns]
attrition                                  object
yearsatcompany                              int64


## Создание флагов аномалий
Помечаем аномалии, **не исключая** строки:
- `review_date_anomaly` — 1, если у сотрудника были опросы, но все они вне допустимого диапазона дат (до приема или после приема + стаж).
- `training_anomaly` — 1, если пройдено тренировок больше, чем положено в год.
- `rating_missing` — 1, если нет валидного опроса вообще (ни одного ревью в допустимом диапазоне).

In [ ]:
all_reviewed = set(performance_clean['employeeid'].unique())
valid_reviewed = set(last_rating['employeeid'].unique())
anomalous_reviewed = all_reviewed - valid_reviewed

df['review_date_anomaly'] = df['employeeid'].isin(anomalous_reviewed).astype(int)
df['training_anomaly'] = (df['trainingopportunitiestaken'] > df['trainingopportunitieswithinyear']).astype(int)
df['rating_missing'] = df['performanceid'].isna().astype(int)

print(f'review_date_anomaly = 1: {df["review_date_anomaly"].sum()}')
print(f'training_anomaly = 1: {df["training_anomaly"].sum()}')
print(f'rating_missing = 1: {df["rating_missing"].sum()}')
print(f'Итого строк: {len(df)}')

review_date_anomaly = 1: 191
training_anomaly = 1: 129
rating_missing = 1: 381
Итого строк: 1470


## Создание новых признаков
Создаем группы по возрасту, зарплате, интегральный индекс удовлетворенности и флаг текучести.

In [ ]:
from datetime import datetime

bins_age = [0, 30, 40, 50, 100]
labels_age = ['до 30', '30-40', '40-50', '50+']
df['age_group'] = pd.cut(df['age'], bins=bins_age, labels=labels_age, right=False)

bins_salary = [0, 50000, 100000, 150000, np.inf]
labels_salary = ['низкая', 'средняя', 'выше среднего', 'высокая']
df['salary_band'] = pd.cut(df['salary'], bins=bins_salary, labels=labels_salary)

sat_cols = ['environmentsatisfaction', 'jobsatisfaction', 'relationshipsatisfaction']
df['satisfaction_score'] = df[sat_cols].mean(axis=1).round(2)

df['attrition_flag'] = (df['attrition'] == 'Yes').astype(int)

print(df[['employeeid', 'age', 'age_group', 'salary', 'salary_band', 'satisfaction_score', 'attrition_flag', 'rating_missing', 'review_date_anomaly', 'training_anomaly']].head(10))

  employeeid  age age_group  salary    salary_band  satisfaction_score  \
0  3012-1A41   30     30-40  102059  выше среднего                4.33   
1  CBCB-9C9D   38     30-40  157718        высокая                4.00   
2  95D7-1CE9   43     40-50  309964        высокая                3.67   
3  47A0-559B   39     30-40  293132        высокая                1.33   
4  42CC-040A   29     до 30   49606         низкая                4.00   
5  C219-6C2E   34     30-40  133468  выше среднего                3.00   
6  D906-B674   42     40-50  259284        высокая                4.33   
7  3C7D-86ED   40     40-50  104426  выше среднего                3.00   
8  3D71-8DC2   38     30-40  147098  выше среднего                3.33   
9  5476-CA0D   31     30-40   69747        средняя                3.00   

   attrition_flag  rating_missing  review_date_anomaly  training_anomaly  
0               0               0                    0                 0  
1               0               0  

## Итоговая сводка по предобработке

In [ ]:
print('=== ИТОГОВАЯ СВОДКА ПРЕДОБРАБОТКИ ===')
print(f'Размер итоговой таблицы: {df.shape}')
print(f'Уникальных сотрудников: {df["employeeid"].nunique()}')
print(f'Сотрудников без валидного опроса (rating_missing=1): {df["rating_missing"].sum()}')
print(f'Аномальная дата опроса (review_date_anomaly=1): {df["review_date_anomaly"].sum()}')
print(f'Аномалия тренингов (training_anomaly=1): {df["training_anomaly"].sum()}')
print(f'Уволившихся (attrition_flag=1): {df["attrition_flag"].sum()} из {len(df)} ({(df["attrition_flag"].mean()*100):.1f}%)')

=== ИТОГОВАЯ СВОДКА ПРЕДОБРАБОТКИ ===
Размер итоговой таблицы: (1470, 46)
Уникальных сотрудников: 1470
Сотрудников без валидного опроса (rating_missing=1): 381
Аномальная дата опроса (review_date_anomaly=1): 191
Аномалия тренингов (training_anomaly=1): 129
Уволившихся (attrition_flag=1): 237 из 1470 (16.1%)


In [ ]:
short_tenure_no_rating = df[(df['yearsatcompany'] < 1) & (df['rating_missing'] == 1)]
print(f'\nСотрудников со стажем < 1 года и без валидного опроса: {len(short_tenure_no_rating)}')


Сотрудников со стажем < 1 года и без валидного опроса: 190


In [ ]:
any_anomaly = df[['review_date_anomaly', 'training_anomaly', 'rating_missing']].any(axis=1)
print(f'Строк с хотя бы одной аномалией: {any_anomaly.sum()} ({(any_anomaly.mean()*100):.1f}%)')

Строк с хотя бы одной аномалией: 510 (34.7%)


## Сохранение результата
Сохраняем итоговую таблицу для последующего анализа гипотез.

In [ ]:
OUTPUT_PATH = 'V4_Employee_cleaned.csv'
df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print('Сохранено в:', OUTPUT_PATH)
print('Размер сохранённой таблицы:', df.shape)

Сохранено в: V4_Employee_cleaned.csv
Размер сохранённой таблицы: (1470, 46)
